In [ ]:
from google.colab import drive
drive.mount("/content/drive")

FOP_ROOT = "/content/drive/MyDrive/FOP"
VIDEO_ROOT = "/content/drive/MyDrive/data/videos"

%cd /content/drive/MyDrive/FOP

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/FOP


In [ ]:
!pip install --no-cache-dir \
  speechbrain \
  facenet-pytorch \
  opencv-python \
  soundfile \
  scipy \
  pandas \
  numpy \
  tqdm

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchcodec

Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: torchvision 0.17.2
Uninstalling torchvision-0.17.2:
  Successfully uninstalled torchvision-0.17.2
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Found existing installation: torchcodec 0.10.0+cu128
Uninstalling torchcodec-0.10.0+cu128:
  Successfully uninstalled torchcodec-0.10.0+cu128


In [ ]:
!pip install --no-cache-dir torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 100.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 119.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 337.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 340.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.6/209.6 MB 203.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 323.0 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nv

In [ ]:
import torch
import torchaudio
import torchvision

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("torchaudio:", torchaudio.__version__)
print("torchvision:", torchvision.__version__)

torch: 2.5.1+cu121
cuda: 12.1
cuda available: True
torchaudio: 2.5.1+cu121
torchvision: 0.20.1+cu121


In [ ]:
!pip install --no-cache-dir speechbrain opencv-python soundfile scipy pandas numpy tqdm imageio-ffmpeg
!pip install --no-cache-dir --no-deps facenet-pytorch==2.6.0

In [ ]:
from speechbrain.inference.speaker import EncoderClassifier
print("SpeechBrain OK")

from facenet_pytorch import MTCNN, InceptionResnetV1
print("FaceNet OK")

SpeechBrain OK
FaceNet OK


In [ ]:
%%writefile pipeline/extract_original_english.py
import argparse
import csv
import json
import os
from pathlib import Path

import cv2
import numpy as np
import soundfile as sf
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from scipy.signal import resample_poly
from speechbrain.inference.speaker import EncoderClassifier


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

_ecapa_model = None
_mtcnn = None
_facenet = None


def get_ecapa():
    global _ecapa_model
    if _ecapa_model is None:
        print("[INFO] Loading SpeechBrain ECAPA-TDNN...")
        _ecapa_model = EncoderClassifier.from_hparams(
            source="speechbrain/spkrec-ecapa-voxceleb",
            savedir="pretrained_models/spkrec-ecapa-voxceleb",
            run_opts={"device": DEVICE},
        )
    return _ecapa_model


def extract_audio_from_video(video_path: str, out_wav: str) -> str:
    os.makedirs(os.path.dirname(out_wav), exist_ok=True)

    import imageio_ffmpeg
    import subprocess

    ffmpeg_bin = imageio_ffmpeg.get_ffmpeg_exe()

    cmd = [
        ffmpeg_bin,
        "-y",
        "-i", video_path,
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        out_wav,
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        raise RuntimeError(
            f"ffmpeg failed for {video_path}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )

    return out_wav


def extract_ecapa(audio_path: str) -> np.ndarray:
    model = get_ecapa()

    audio_np, sample_rate = sf.read(audio_path, dtype="float32", always_2d=False)

    if sample_rate != 16000:
        import math
        gcd = math.gcd(16000, sample_rate)
        audio_np = resample_poly(
            audio_np,
            16000 // gcd,
            sample_rate // gcd,
        ).astype(np.float32)

    waveform = torch.from_numpy(audio_np).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        emb = model.encode_batch(waveform)

    return emb.squeeze().cpu().numpy()


def get_facenet():
    global _mtcnn, _facenet

    if _mtcnn is None:
        print("[INFO] Loading MTCNN...")
        _mtcnn = MTCNN(
            image_size=160,
            margin=20,
            keep_all=False,
            post_process=True,
            device=DEVICE,
        )

    if _facenet is None:
        print("[INFO] Loading FaceNet...")
        _facenet = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)

    return _mtcnn, _facenet


def extract_facenet(video_path: str, frame_step: int = 25):
    mtcnn, facenet = get_facenet()

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    results = []
    frame_idx = 0

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        face_tensor = mtcnn(frame_rgb)

        if face_tensor is not None:
            with torch.no_grad():
                emb = facenet(face_tensor.unsqueeze(0).to(DEVICE))
            results.append((frame_idx, emb.squeeze().cpu().numpy()))

        frame_idx += 1

    cap.release()
    return results


def rel_path(path: Path, root: Path) -> str:
    return "./" + str(path.resolve().relative_to(root.resolve())).replace("\\", "/")


def save_ecapa(emb, fop_root, speaker_id, language, video_id):
    out_dir = (
        Path(fop_root)
        / "ecappafeats_original_extracted"
        / "v3"
        / "voices"
        / speaker_id
        / language
        / video_id
    )
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "00000.npy"
    np.save(out_path, emb)

    return out_path


def save_facenet(face_embs, fop_root, speaker_id, language, video_id):
    out_dir = (
        Path(fop_root)
        / "facenetfeats_original_extracted"
        / "v3"
        / "faces"
        / speaker_id
        / language
        / video_id
    )
    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []

    for frame_idx, emb in face_embs:
        out_path = out_dir / f"{frame_idx:09d}.npy"
        np.save(out_path, emb)
        saved.append(out_path)

    return saved


def process_video(video_path, fop_root, speaker_id, label, language, frame_step):
    video_path = Path(video_path)
    video_id = f"{video_path.parent.name}_{video_path.stem}"

    print(f"[INFO] Processing {speaker_id} / {language} / {video_id}")

    work_dir = Path(fop_root) / "pipeline_output_original" / speaker_id / video_id
    work_dir.mkdir(parents=True, exist_ok=True)

    audio_wav = work_dir / "original.wav"

    extract_audio_from_video(str(video_path), str(audio_wav))

    audio_emb = extract_ecapa(str(audio_wav))
    ecapa_path = save_ecapa(audio_emb, fop_root, speaker_id, language, video_id)

    face_embs = extract_facenet(str(video_path), frame_step=frame_step)

    if len(face_embs) == 0:
        print(f"[WARNING] No face detected every {frame_step} frames. Retrying every 2 frames.")
        face_embs = extract_facenet(str(video_path), frame_step=2)

    if len(face_embs) == 0:
        raise RuntimeError(f"No face embeddings extracted from {video_path}")

    facenet_paths = save_facenet(face_embs, fop_root, speaker_id, language, video_id)

    rows = []
    root = Path(fop_root)

    for face_path in facenet_paths:
        rows.append([
            "",
            "",
            speaker_id,
            rel_path(face_path, root),
            int(label),
            rel_path(ecapa_path, root),
            "",
        ])

    return rows


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--video_dir", required=True)
    parser.add_argument("--fop_root", required=True)
    parser.add_argument("--speaker_map", required=True)
    parser.add_argument("--language", default="English")
    parser.add_argument("--out_csv", default="feature_tracker/v3_train_English_original_extracted.csv")
    parser.add_argument("--frame_step", type=int, default=25)
    parser.add_argument("--max_videos_per_speaker", type=int, default=0)

    args = parser.parse_args()

    video_root = Path(args.video_dir)
    fop_root = Path(args.fop_root)

    with open(args.speaker_map, "r") as f:
        speaker_map = json.load(f)

    all_rows = []
    errors = []

    for speaker_id, label in speaker_map.items():
        source_dir = video_root / speaker_id / args.language

        if not source_dir.exists():
            print(f"[WARNING] Missing folder: {source_dir}")
            continue

        videos = sorted(
            list(source_dir.rglob("*.avi")) +
            list(source_dir.rglob("*.mp4"))
        )

        if args.max_videos_per_speaker > 0:
            videos = videos[:args.max_videos_per_speaker]

        print(f"\n[INFO] Speaker {speaker_id}: {len(videos)} videos")

        for video_path in videos:
            try:
                rows = process_video(
                    video_path=video_path,
                    fop_root=fop_root,
                    speaker_id=speaker_id,
                    label=label,
                    language=args.language,
                    frame_step=args.frame_step,
                )
                all_rows.extend(rows)

            except Exception as exc:
                print(f"[ERROR] Failed {speaker_id} / {video_path}: {exc}")
                errors.append({
                    "speaker_id": speaker_id,
                    "video": str(video_path),
                    "error": str(exc),
                })

    out_csv = fop_root / args.out_csv
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    header = [
        "audio_path",
        "face_path",
        "identity",
        "facenet_feats_path",
        "label",
        "ecappa_feats_path",
        "wavlmsv_feats_path",
    ]

    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(all_rows)

    print(f"\n[INFO] Saved CSV: {out_csv}")
    print(f"[INFO] Rows: {len(all_rows)}")
    print(f"[INFO] Errors: {len(errors)}")

    if errors:
        error_path = fop_root / "pipeline_output_original" / "original_english_extraction_errors.json"
        error_path.parent.mkdir(parents=True, exist_ok=True)

        with open(error_path, "w") as f:
            json.dump(errors, f, indent=2)

        print(f"[INFO] Errors saved to: {error_path}")


if __name__ == "__main__":
    main()

Writing pipeline/extract_original_english.py


In [ ]:
!python pipeline/extract_original_english.py \
  --video_dir "/content/drive/MyDrive/data/videos" \
  --fop_root "/content/drive/MyDrive/FOP" \
  --speaker_map "/content/drive/MyDrive/FOP/speaker_map.json" \
  --language English \
  --out_csv "feature_tracker/v3_train_English_original_extracted_test.csv" \
  --frame_step 25 \
  --max_videos_per_speaker 1


[INFO] Speaker id0001: 1 videos
[INFO] Processing id0001 / English / XtKYDOsvsG0_00000
[INFO] Loading SpeechBrain ECAPA-TDNN...
/usr/local/lib/python3.12/dist-packages/speechbrain/utils/checkpoints.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/FOP/feature_tracker/v3_train_English_original_extracted_test.csv")
print(df.shape)
print(df["identity"].nunique())
df.head()

(476, 7)
36


,audio_path,face_path,identity,facenet_feats_path,label,ecappa_feats_path,wavlmsv_feats_path
0,NaN,NaN,id0001,./facenetfeats_original_extracted/v3/faces/id0...,0,./ecappafeats_original_extracted/v3/voices/id0...,NaN
1,NaN,NaN,id0001,./facenetfeats_original_extracted/v3/faces/id0...,0,./ecappafeats_original_extracted/v3/voices/id0...,NaN
2,NaN,NaN,id0001,./facenetfeats_original_extracted/v3/faces/id0...,0,./ecappafeats_original_extracted/v3/voices/id0...,NaN
3,NaN,NaN,id0001,./facenetfeats_original_extracted/v3/faces/id0...,0,./ecappafeats_original_extracted/v3/voices/id0...,NaN
4,NaN,NaN,id0001,./facenetfeats_original_extracted/v3/faces/id0...,0,./ecappafeats_original_extracted/v3/voices/id0...,NaN


In [ ]:
!python pipeline/extract_original_english.py \
  --video_dir "/content/drive/MyDrive/data/videos" \
  --fop_root "/content/drive/MyDrive/FOP" \
  --speaker_map "/content/drive/MyDrive/FOP/speaker_map.json" \
  --language English \
  --out_csv "feature_tracker/v3_train_English_original_extracted.csv" \
  --frame_step 25


[INFO] Speaker id0001: 9 videos
[INFO] Processing id0001 / English / XtKYDOsvsG0_00000
[INFO] Loading SpeechBrain ECAPA-TDNN...
/usr/local/lib/python3.12/dist-packages/speechbrain/utils/checkpoints.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please

In [ ]:
import pandas as pd
import os

FOP_ROOT = "/content/drive/MyDrive/FOP"

df = pd.read_csv(FOP_ROOT + "/feature_tracker/v3_train_English_original_extracted.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Identities:", df["identity"].nunique())
print("Labels:", df["label"].nunique())
print("Min label:", df["label"].min())
print("Max label:", df["label"].max())

# Check first paths
row = df.iloc[0]
audio_path = os.path.join(FOP_ROOT, row["ecappa_feats_path"].replace("./", ""))
face_path = os.path.join(FOP_ROOT, row["facenet_feats_path"].replace("./", ""))

print("Audio exists:", os.path.exists(audio_path), audio_path)
print("Face exists:", os.path.exists(face_path), face_path)

Rows: 16431
Columns: ['audio_path', 'face_path', 'identity', 'facenet_feats_path', 'label', 'ecappa_feats_path', 'wavlmsv_feats_path']
Identities: 36
Labels: 36
Min label: 0
Max label: 35
Audio exists: True /content/drive/MyDrive/FOP/ecappafeats_original_extracted/v3/voices/id0001/English/XtKYDOsvsG0_00000/00000.npy
Face exists: True /content/drive/MyDrive/FOP/facenetfeats_original_extracted/v3/faces/id0001/English/XtKYDOsvsG0_00000/000000000.npy


In [ ]:
import re
import pandas as pd
from pathlib import Path

#FOP_ROOT = r"C:\Users\beiba\Desktop\baseline-polysim\Poly-sim\FOP"

official_test_csv = Path(FOP_ROOT) / "feature_tracker" / "v3_test_English.csv"
extracted_csv = Path(FOP_ROOT) / "feature_tracker" / "v3_train_English_original_extracted.csv"

out_csv = Path(FOP_ROOT) / "feature_tracker" / "v3_test_English_original_extracted.csv"

official = pd.read_csv(official_test_csv)
extracted = pd.read_csv(extracted_csv)


def normalize_path(p):
    return str(p).replace("\\", "/")


def get_key_from_official_ecappa(path):
    """
    Example official path:
    ./ecappafeats/v3/voices/id0001/English/h_vamljclHE/00000.npy

    Returns:
    (id0001, h_vamljclHE, 00000)
    """
    path = normalize_path(path)

    m = re.search(
        r"voices/([^/]+)/English/([^/]+)/([^/]+)\.npy",
        path
    )

    if m is None:
        return None

    identity, video_id, clip_id = m.groups()
    return identity, video_id, clip_id


def get_key_from_extracted_ecappa(path):
    """
    Example extracted path:
    ./ecappafeats_original_extracted/v3/voices/id0001/English/h_vamljclHE_00000/00000.npy

    Returns:
    (id0001, h_vamljclHE, 00000)
    """
    path = normalize_path(path)

    m = re.search(
        r"voices/([^/]+)/English/([^/]+)/00000\.npy",
        path
    )

    if m is None:
        return None

    identity, extracted_video_id = m.groups()

    # extracted_video_id is expected to be video_clip, e.g. h_vamljclHE_00000
    if "_" in extracted_video_id:
        video_id, clip_id = extracted_video_id.rsplit("_", 1)
    else:
        # fallback if no clip suffix exists
        video_id = extracted_video_id
        clip_id = "00000"

    return identity, video_id, clip_id


# Build official test clip keys
official["clip_key"] = official["ecappa_feats_path"].apply(get_key_from_official_ecappa)
official_keys = set(k for k in official["clip_key"] if k is not None)

print("Official test rows:", len(official))
print("Official unique test clips:", len(official_keys))

# Build extracted clip keys
extracted["clip_key"] = extracted["ecappa_feats_path"].apply(get_key_from_extracted_ecappa)

# Keep only extracted rows that match official test clips
matched = extracted[extracted["clip_key"].isin(official_keys)].copy()

# Remove helper column
matched = matched.drop(columns=["clip_key"])

matched.to_csv(out_csv, index=False)

print("Extracted rows:", len(extracted))
print("Matched extracted rows:", len(matched))
print("Matched identities:", matched["identity"].nunique())
print("Saved:", out_csv)

Official test rows: 232
Official unique test clips: 226
Extracted rows: 16431
Matched extracted rows: 3203
Matched identities: 36
Saved: /content/drive/MyDrive/FOP/feature_tracker/v3_test_English_original_extracted.csv


In [ ]:
import pandas as pd
from pathlib import Path

#FOP_ROOT = r"C:\Users\beiba\Desktop\baseline-polysim\Poly-sim\FOP"

official = pd.read_csv(Path(FOP_ROOT) / "feature_tracker" / "v3_test_English.csv")
matched = pd.read_csv(Path(FOP_ROOT) / "feature_tracker" / "v3_test_English_original_extracted.csv")

print("Official test:", official.shape, official["identity"].nunique())
print("Extracted matched test:", matched.shape, matched["identity"].nunique())
print(matched.head())

Official test: (232, 7) 36
Extracted matched test: (3203, 7) 36
   audio_path  face_path identity  \
0         NaN        NaN   id0001   
1         NaN        NaN   id0001   
2         NaN        NaN   id0001   
3         NaN        NaN   id0001   
4         NaN        NaN   id0001   

                                  facenet_feats_path  label  \
0  ./facenetfeats_original_extracted/v3/faces/id0...      0   
1  ./facenetfeats_original_extracted/v3/faces/id0...      0   
2  ./facenetfeats_original_extracted/v3/faces/id0...      0   
3  ./facenetfeats_original_extracted/v3/faces/id0...      0   
4  ./facenetfeats_original_extracted/v3/faces/id0...      0   

                                   ecappa_feats_path  wavlmsv_feats_path  
0  ./ecappafeats_original_extracted/v3/voices/id0...                 NaN  
1  ./ecappafeats_original_extracted/v3/voices/id0...                 NaN  
2  ./ecappafeats_original_extracted/v3/voices/id0...                 NaN  
3  ./ecappafeats_original_extracte